In [ ]:
import numpy as np
import pandas as pd
import torch


from lstm_dos_rewritten import lstm_seq2seq


# =========================
# CONFIG
# =========================
TRAIN_CSV = "data/vessel_prediction_model_data/mid_model_data/mid_train_data.csv"
VAL_CSV   = "data/vessel_prediction_model_data/mid_model_data/mid_val_data.csv"
TEST_CSV  = "data/vessel_prediction_model_data/mid_model_data/mid_test_data.csv"

SEQ_LEN = 5
TARGET_LEN = 1

BATCH_SIZE = 64
HIDDEN_SIZE = 128
NUM_LAYERS = 2
DROPOUT = 0.2
LEARNING_RATE = 0.001
MAX_EPOCHS = 100
TEACHER_FORCING_RATIO = 0.7

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SAVE_PATH = "best_lstm_seq2seq.pt"


# =========================
# CHOOSE COLUMNS
# =========================
feature_cols = [
    "LAT",
    "LON",
    "SPEED",
    "dt",
    "cog_sin",
    "cog_cos",
    "hdg_sin",
    "hdg_cos",
    "dlat",
    "dlon",
]

target_cols = ["dlat", "dlon"]


# =========================
# LOAD DATA
# =========================
train_df = pd.read_csv(TRAIN_CSV)
val_df   = pd.read_csv(VAL_CSV)
test_df  = pd.read_csv(TEST_CSV)


# =========================
# SCALE FEATURES USING TRAIN ONLY
# =========================
train_means = train_df[feature_cols].mean()
train_stds = train_df[feature_cols].std().replace(0, 1.0)

def scale_df(df: pd.DataFrame, means: pd.Series, stds: pd.Series) -> pd.DataFrame:
    df = df.copy()
    df[feature_cols] = (df[feature_cols] - means) / stds
    return df

train_df = scale_df(train_df, train_means, train_stds)
val_df   = scale_df(val_df, train_means, train_stds)
test_df  = scale_df(test_df, train_means, train_stds)


# =========================
# WINDOW FUNCTION
# =========================
def make_windows_from_df(
    df: pd.DataFrame,
    feature_cols,
    target_cols,
    seq_len,
    target_len,
    mmsi_col: str = "MMSI",
    time_col: str = "TIME"
):
    """
    Returns:
      X: torch.FloatTensor  (seq_len,  N, n_features)
      Y: torch.FloatTensor  (target_len, N, n_targets)
    """
    df = df.copy()
    df[time_col] = pd.to_datetime(df[time_col])

    X_list = []
    Y_list = []

    for mmsi, g in df.groupby(mmsi_col):
        g = g.sort_values(time_col).reset_index(drop=True)

        feats = g[feature_cols].to_numpy(dtype=np.float32)
        targs = g[target_cols].to_numpy(dtype=np.float32)

        n = len(g)
        if n < seq_len + target_len:
            continue

        for i in range(n - (seq_len + target_len) + 1):
            X_list.append(feats[i : i + seq_len])
            Y_list.append(targs[i + seq_len : i + seq_len + target_len])

    if len(X_list) == 0:
        raise ValueError("No windows created. Increase data or reduce seq_len/target_len.")

    X = torch.from_numpy(np.stack(X_list, axis=0))  # (N, seq_len, n_features)
    Y = torch.from_numpy(np.stack(Y_list, axis=0))  # (N, target_len, n_targets)

    X = X.permute(1, 0, 2).contiguous()  # (seq_len, N, n_features)
    Y = Y.permute(1, 0, 2).contiguous()  # (target_len, N, n_targets)

    return X, Y


X_train, Y_train = make_windows_from_df(
    train_df, feature_cols, target_cols, seq_len=SEQ_LEN, target_len=TARGET_LEN
)
X_val, Y_val = make_windows_from_df(
    val_df, feature_cols, target_cols, seq_len=SEQ_LEN, target_len=TARGET_LEN
)
X_test, Y_test = make_windows_from_df(
    test_df, feature_cols, target_cols, seq_len=SEQ_LEN, target_len=TARGET_LEN
)

print("X_train:", X_train.shape)
print("Y_train:", Y_train.shape)
print("X_val:  ", X_val.shape)
print("Y_val:  ", Y_val.shape)
print("X_test: ", X_test.shape)
print("Y_test: ", Y_test.shape)


# =========================
# MODEL
# =========================
model = lstm_seq2seq(
    input_size=len(feature_cols),
    hidden_size=HIDDEN_SIZE,
    output_size=2,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT,
    target_indices=(0, 1),
    decoder_feature_indices=tuple(range(len(feature_cols))),
    predict_deltas=True,
).to(DEVICE)

print(model)


# =========================
# TRAIN
# =========================
history = model.train_model(
    input_tensor=X_train,
    target_tensor=Y_train,
    target_len=TARGET_LEN,
    batch_size=BATCH_SIZE,
    val_input_tensor=X_val,
    val_target_tensor=Y_val,
    max_epochs=MAX_EPOCHS,
    training_prediction="mixed_teacher_forcing",
    teacher_forcing_ratio=TEACHER_FORCING_RATIO,
    learning_rate=LEARNING_RATE,
    dynamic_tf=True,
    patience=10,
    min_delta=1e-5,
    save_path=SAVE_PATH,
    grad_clip=1.0,
    shuffle_batches=True,
)

print("\nTraining done.")
print("Best epoch:", history["best_epoch"])
print("Best val loss:", history["best_val_loss"])


# =========================
# LOAD BEST MODEL
# =========================
model.load_state_dict(torch.load(SAVE_PATH, map_location=DEVICE))
model.eval()


# =========================
# EVALUATION ON TEST SET
# =========================
def predict_dataset(model, X_tensor, target_len, batch_size=256):
    """
    Returns predictions in absolute coordinates:
    pred shape: (target_len, N, 2)
    """
    model.eval()
    preds = []

    n_samples = X_tensor.shape[1]

    with torch.no_grad():
        for start in range(0, n_samples, batch_size):
            end = min(start + batch_size, n_samples)
            x_batch = X_tensor[:, start:end, :].to(DEVICE)

            # Run recursive decoder rollout directly
            _, encoder_hidden = model.encoder(x_batch)
            decoder_input = model._make_initial_decoder_input(x_batch)
            outputs = model._decode_rollout_recursive(
                decoder_input, encoder_hidden, target_len, x_batch
            )

            if model.predict_deltas:
                outputs = model._outputs_to_absolute(x_batch, outputs)

            preds.append(outputs.cpu())

    return torch.cat(preds, dim=1)


Y_pred_test = predict_dataset(model, X_test, target_len=TARGET_LEN, batch_size=256)

print("Y_pred_test:", Y_pred_test.shape)


# =========================
# METRICS
# =========================
def rmse_mae(pred, true):
    """
    pred, true: torch tensors of shape (target_len, N, 2)
    """
    err = pred - true
    mse = torch.mean(err ** 2).item()
    rmse = torch.sqrt(torch.mean(err ** 2)).item()
    mae = torch.mean(torch.abs(err)).item()
    return mse, rmse, mae


mse, rmse, mae = rmse_mae(Y_pred_test, Y_test)

print("\nTest metrics:")
print("MSE: ", mse)
print("RMSE:", rmse)
print("MAE: ", mae)


# =========================
# EXAMPLE SINGLE-SAMPLE PREDICTION
# =========================
sample_idx = 0
x_sample = X_test[:, sample_idx, :].to(DEVICE)   # (seq_len, features)

pred_sample = model.predict(
    x_sample,
    target_len=TARGET_LEN,
    return_absolute=True
)

true_sample = Y_test[:, sample_idx, :].cpu().numpy()

print("\nSample prediction:")
print("Predicted:")
print(pred_sample)

print("\nTrue:")
print(true_sample)

X_train: torch.Size([5, 27710, 10])
Y_train: torch.Size([1, 27710, 2])
X_val:   torch.Size([5, 6198, 10])
Y_val:   torch.Size([1, 6198, 2])
X_test:  torch.Size([5, 2755, 10])
Y_test:  torch.Size([1, 2755, 2])
lstm_seq2seq(
  (encoder): lstm_encoder(
    (lstm): LSTM(10, 128, num_layers=2, dropout=0.2)
  )
  (decoder): lstm_decoder(
    (lstm): LSTM(10, 128, num_layers=2, dropout=0.2)
    (linear): Linear(in_features=128, out_features=2, bias=True)
  )
  (target_to_decoder): Linear(in_features=2, out_features=10, bias=True)
)


  0%|          | 0/100 [00:00<?, ?it/s]

In [ ]:
import folium

def mapmake(history, prediction):
    m = folium.Map(location=[center_lat, center_lon], zoom_start=10)

    # History points
    history_coords = [(row[0], row[1]) for row in history]

    # Predicted points
    pred_coords = [(row[0], row[1]) for row in prediction]

    # Draw history track
    folium.PolyLine(
        history_coords,
        weight=3,
        tooltip="History"
    ).add_to(m)

    # Add last point marker
    folium.Marker(
        location=list(history_coords[-1]),
        popup="Last ping",
        tooltip="Last ping"
    ).add_to(m)

    # Draw predicted track
    folium.PolyLine(
        pred_coords,
        weight=3,
        dash_array="5, 8",
        tooltip="Predicted"
    ).add_to(m)

    # Add markers for history
    for i, (lat, lon) in enumerate(history_coords):
        folium.CircleMarker(
            location=[lat, lon],
            radius=3,
            popup=f"History {i}",
            fill=True
        ).add_to(m)

    # Add markers for predicted points
    for i, (lat, lon) in enumerate(pred_coords, start=1):
        folium.Marker(
            location=[lat, lon],
            popup=f"Predicted {i}",
            tooltip=f"Predicted {i}"
        ).add_to(m)

    return m

In [ ]:
# converts mse into meters, then yards
def haversine_m(lat1, lon1, lat2, lon2):
    R = 6371000  # meters

    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = np.sin(dlat / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2)**2
    c = 2 * np.arcsin(np.sqrt(a))

    return R * c  # meters

# meters to yards
def mse_yards(pred, true):
    """
    pred, true: (target_len, N, 2)
    """

    pred_lat = pred[..., 0]
    pred_lon = pred[..., 1]
    true_lat = true[..., 0]
    true_lon = true[..., 1]

    d_m = haversine_m(true_lat, true_lon, pred_lat, pred_lon)

    d_yards = d_m * 1.09361

    mse = np.mean(d_yards ** 2)

    return mse

def rmse_yards(pred, true):
    mse = mse_yards(pred, true)
    return np.sqrt(mse)

In [ ]:
pred_np = Y_pred_test.cpu().numpy()
true_np = Y_test.cpu().numpy()

mse_val = mse_yards(pred_np, true_np)
print("MSE (yards^2):", mse_val)

MSE (yards^2): 5.905656e+09


In [ ]:
idx = 2

# 1. Get sample from test set
history = X_test[:, idx, :]   # (seq_len, features)

# 2. Convert to numpy
history_np = history.cpu().numpy()

# 3. Extract LAT/LON (still SCALED right now)
history_latlon = history_np[:, [0, 1]].copy()

# 4. Unscale LAT/LON
history_latlon[:, 0] = history_latlon[:, 0] * train_stds["LAT"] + train_means["LAT"]
history_latlon[:, 1] = history_latlon[:, 1] * train_stds["LON"] + train_means["LON"]

# 5. Get predicted deltas (still SCALED)
pred_delta = model.predict(
    history.to(DEVICE),
    target_len=TARGET_LEN,
    return_absolute=False
)

# 6. Unscale predicted deltas
pred_delta_unscaled = pred_delta.copy()

pred_delta_unscaled[:, 0] = pred_delta[:, 0] * target_stds["dlat"] + target_means["dlat"]
pred_delta_unscaled[:, 1] = pred_delta[:, 1] * target_stds["dlon"] + target_means["dlon"]

# 7. Convert deltas → absolute lat/lon
current = history_latlon[-1].copy()
pred_abs = []

for dlat, dlon in pred_delta_unscaled:
    current = current + np.array([dlat, dlon])
    pred_abs.append(current.copy())

pred_abs = np.array(pred_abs)

# 8. Set map center
center_lat = history_latlon[-1, 0]
center_lon = history_latlon[-1, 1]

# 9. Plot
m = mapmake(history_latlon, pred_abs)
m